# Word Mover’s Distance(WMD)

NLP연구쪽에서 문서 유사도 구하는 방법으로 잘 쓰인다고 함. (논문 연구 결과 다른 방법론보다 성능이 좋다고 함)\
기본적으로 word2vec을 베이스로 하고, word2vec을 사용해서 word간의 euclide distance를 구함!

- 피험자 1명, seed 단어 1개

피험자가 대답한 30개의 연속적인 단어 그룹 - target word(money, friend, family)의 유사도를 기반으로 wmdistance를 잰다.

--> 그럼 3개의 distance값이 나옴. 그 중 가장 값이 작은 것이 거리가 가깝다. 즉, target word와 유사하다.

wmd 설명 참조
- https://sy-programmingstudy.tistory.com/14
- https://www.youtube.com/watch?v=zFnrq5SmBdg

In [1]:
import os

import pandas as pd
from gensim.models import KeyedVectors

In [2]:
# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
pilot2_dir = os.getcwd()
# data/processed 폴더 위치 지정
processed_data_dir = pilot2_dir + '/data/processed/' 

In [3]:
# word2vec model 로딩
word2vec_model = KeyedVectors.load_word2vec_format(pilot2_dir + '/../pretrained/GoogleNews-vectors-negative300.bin', binary=True)

In [15]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
pilot_data = pd.read_csv(processed_data_dir + 'similarity_coherence_data_300_with_words.csv', index_col=0, keep_default_na=False)
pilot_data[0:3]

,subject,key1,key2,key3,key4,key5,key6,key7,key8,key9,...,similarity_friend30_friend,coherence_key_key,coherence_key_money,coherence_key_friend,coherence_money_key,coherence_money_money,coherence_money_friend,coherence_friend_key,coherence_friend_money,coherence_friend_friend
Prolific_ID,,,,,,,,,,,,,,,,,,,,,
5d53bffa147a7d00015aae5a,1,door,gate,outside,grass,itchy,rash,chicken pox,shingles,roofing,...,0.19990666,0.041143,0.102066,0.075201,0.030420,0.143791,0.063515,0.037806,0.096950,0.146998
5f00ec86304f7322eb8dfa41,2,unlock,door,enter,dreams,time space,possibilites,infinite,time,goals,...,0.17192948,0.096018,0.140075,0.120077,0.053525,0.126333,0.125166,0.081238,0.108360,0.105842
5de27ced22383629b807cc70,3,hole,ground,hog,pig,pork,said,dressing,clothes,old,...,0.058080602,0.079032,0.117155,0.169555,0.057519,0.099424,0.132149,0.049724,0.145488,0.082121


In [16]:
seed_words = ['key', 'money', 'friend'] # 정해진 시드 단어들을 배열에 저장
n_respond_words = 30 # 하나의 시드당 30개의 단어 응답
n_subject = len(pilot_data) # 58명의 피험자

In [17]:
# coherence_wmd_* 컬럼 미리 생성(빈 값)
for seed_word in seed_words:
    for target_word in seed_words:
        column_name = f'coherence_wmd_{seed_word}_{target_word}'
        pilot_data[column_name] = 0
        # pilot_data[column_name] = pilot_data[column_name].astype(float)

pilot_data.columns


Index(['subject', 'key1', 'key2', 'key3', 'key4', 'key5', 'key6', 'key7',
       'key8', 'key9',
       ...
       'coherence_friend_friend', 'coherence_wmd_key_key',
       'coherence_wmd_key_money', 'coherence_wmd_key_friend',
       'coherence_wmd_money_key', 'coherence_wmd_money_money',
       'coherence_wmd_money_friend', 'coherence_wmd_friend_key',
       'coherence_wmd_friend_money', 'coherence_wmd_friend_friend'],
      dtype='object', length=379)

In [18]:
# 응답 단어 컬럼 목록 (예: 'key1', 'key2', ... , 'key30', 'money1', 'money2', ... , 'money30', 'friend1', 'friend2', ... , 'friend30')
word_columns = [f'{seed_word}{i}' for seed_word in seed_words for i in range(1, n_respond_words + 1)]
word_columns

['key1',
 'key2',
 'key3',
 'key4',
 'key5',
 'key6',
 'key7',
 'key8',
 'key9',
 'key10',
 'key11',
 'key12',
 'key13',
 'key14',
 'key15',
 'key16',
 'key17',
 'key18',
 'key19',
 'key20',
 'key21',
 'key22',
 'key23',
 'key24',
 'key25',
 'key26',
 'key27',
 'key28',
 'key29',
 'key30',
 'money1',
 'money2',
 'money3',
 'money4',
 'money5',
 'money6',
 'money7',
 'money8',
 'money9',
 'money10',
 'money11',
 'money12',
 'money13',
 'money14',
 'money15',
 'money16',
 'money17',
 'money18',
 'money19',
 'money20',
 'money21',
 'money22',
 'money23',
 'money24',
 'money25',
 'money26',
 'money27',
 'money28',
 'money29',
 'money30',
 'friend1',
 'friend2',
 'friend3',
 'friend4',
 'friend5',
 'friend6',
 'friend7',
 'friend8',
 'friend9',
 'friend10',
 'friend11',
 'friend12',
 'friend13',
 'friend14',
 'friend15',
 'friend16',
 'friend17',
 'friend18',
 'friend19',
 'friend20',
 'friend21',
 'friend22',
 'friend23',
 'friend24',
 'friend25',
 'friend26',
 'friend27',
 'friend28',
 'f

In [19]:

[pilot_data.iloc[0][column] for column in ['key1',
 'key2',
 'key3',
 'key4',
 'key5',
 'key6',
 'key7',
 'key8',
 'key9',
 'key10',
 'key11',
 'key12',
 'key13',
 'key14',
 'key15',
 'key16',
 'key17',
 'key18',
 'key19',
 'key20',
 'key21',
 'key22',
 'key23',
 'key24',
 'key25',
  'key26',
  'key27',
  'key28',
  'key29',
  'key30'] if column.startswith('key')] 

['door',
 'gate',
 'outside',
 'grass',
 'itchy',
 'rash',
 'chicken pox',
 'shingles',
 'roofing',
 'high',
 'afraid',
 'scary',
 'horror',
 'movies',
 'new',
 'games',
 'playstation',
 'sony',
 'walkman',
 'cd player',
 'computer',
 'laptop',
 'typing',
 'class',
 'desks',
 'students',
 'backpacks',
 'notebooks',
 'pencils',
 'drawing']

In [20]:
for i_subject in range(n_subject):
    print('subject: ', i_subject+1)

    for seed_word in seed_words:
        print('seed: ', seed_word)

        # 각 피험자의 응답 문서를 30개씩 단어 리스트로 변환
        key_response_words = [pilot_data.iloc[i_subject][column] for column in word_columns if column.startswith('key')] # 30개 응답단어
        friend_response_words = [pilot_data.iloc[i_subject][column] for column in word_columns if column.startswith('friend')] # 30개 응답단어
        money_response_words = [pilot_data.iloc[i_subject][column] for column in word_columns if column.startswith('money')] # 30개 응답단어

        try:
            # WMD 거리 계산
            key_wmd_distance = word2vec_model.wmdistance([seed_word], key_response_words) # 응답 30개 단어 - key
            friend_wmd_distance = word2vec_model.wmdistance([seed_word], friend_response_words)  # 응답 30개 단어 - friend
            money_wmd_distance = word2vec_model.wmdistance([seed_word], money_response_words)  # 응답 30개 단어 - money
            # 계산한 WMD를 해당 테이블 위치에 저장
            pilot_data.at[i_subject, f'coherence_wmd_{seed_word}_key'] = key_wmd_distance
            pilot_data.at[i_subject, f'coherence_wmd_{seed_word}_friend'] = friend_wmd_distance
            pilot_data.at[i_subject, f'coherence_wmd_{seed_word}_money'] = money_wmd_distance
            print(f'coherence_wmd_{seed_word}_key, coherence_wmd_{seed_word}_friend, coherence_wmd_{seed_word}_money')
            print(f'key: {key_wmd_distance}, friend: {friend_wmd_distance}, money: {money_wmd_distance}')
        except Exception as e:
            print(f"An error occurred for subject {i_subject}: {str(e)}")
            continue


subject:  1
seed:  key
coherence_wmd_key_key, coherence_wmd_key_friend, coherence_wmd_key_money
key: 1.3838922227436379, friend: 1.3863407995830246, money: 1.3919530165556855
seed:  money
coherence_wmd_money_key, coherence_wmd_money_friend, coherence_wmd_money_money
key: 1.3395049448421927, friend: 1.3434402480968382, money: 1.3048113331692182
seed:  friend
coherence_wmd_friend_key, coherence_wmd_friend_friend, coherence_wmd_friend_money
key: 1.35916976996165, friend: 1.305232927878808, money: 1.3674888038162294
subject:  2
seed:  key
coherence_wmd_key_key, coherence_wmd_key_friend, coherence_wmd_key_money
key: 1.343760710597814, friend: 1.3549096788266202, money: 1.3752939848744983
seed:  money
coherence_wmd_money_key, coherence_wmd_money_friend, coherence_wmd_money_money
key: 1.3100134344493806, friend: 1.334133409515669, money: 1.3209716026414686
seed:  friend
coherence_wmd_friend_key, coherence_wmd_friend_friend, coherence_wmd_friend_money
key: 1.3249207099597418, friend: 1.3364242

In [9]:
# # 단어 있는 버전 csv 저장
# pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data_300_with_words_2.csv', index=None)

# # 단어 컬럼들 드롭
# drop_columns = pilot_data.columns[1:91]
# pilot_data = pilot_data.drop(drop_columns, axis='columns')

# # 단어 없이 coherence만 있는 버전 csv 저장
# pilot_data.to_csv(processed_data_dir + 'similarity_coherence_data_300_2.csv', index=None)